# Car Model Detection — Kaggle training run

Runs the full pipeline on Kaggle free GPU (T4/P100): data → linear probe → fine-tune → evaluate → ONNX export.

**Setup (once):**
1. Settings → Accelerator → **GPU**, Internet → **On**.
2. Either set `REPO_URL` below to your GitHub repo, **or** upload the repo's `ml/` folder + `pyproject.toml` as a Kaggle Dataset and adjust `REPO_DIR`.

**Resume after a session cap:** just re-run all cells — `--resume auto` picks up from `last.ckpt` in `/kaggle/working/runs/` (persisted if you save a notebook version, otherwise download `runs/` before the session dies).

In [ ]:
%pip install -q timm onnx onnxruntime
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/SHARVESH08/car-game.git"
REPO_DIR = "/kaggle/working/car-game"
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("repo at", os.getcwd())

In [ ]:
# Stanford Cars via the torchvision-compatible Kaggle mirror
# (original Stanford URLs are dead — pytorch/vision#7545)
import kagglehub
DATA_ROOT = kagglehub.dataset_download("rickyyyyyyy/torchvision-stanford-cars")
print("data at", DATA_ROOT)
import os
print(os.listdir(DATA_ROOT))  # expect: stanford_cars/

In [ ]:
# Emit classes.json / labels.json / test_manifest.json (also sanity-checks the data)
!python -m ml.dataset --data-root "$DATA_ROOT"

In [ ]:
# Stage 1 — linear probe (cheap baseline, ~15 min). Sets the bar.
!python -m ml.train --data-root "$DATA_ROOT" --mode linear_probe --epochs 5 \
    --out-dir /kaggle/working/runs/probe --resume auto

In [ ]:
# Stage 2 — full fine-tune (~2-4 h on T4). Safe to interrupt: resumes from last.ckpt.
!python -m ml.train --data-root "$DATA_ROOT" --mode finetune --epochs 30 \
    --out-dir /kaggle/working/runs/finetune --resume auto

In [ ]:
# Evaluate on the FROZEN official test split — these are the published numbers.
!python -m ml.evaluate --data-root "$DATA_ROOT" \
    --checkpoint /kaggle/working/runs/finetune/best.pt

In [ ]:
# ONNX export + torch/onnx parity check + CPU latency benchmark
!python -m ml.export_onnx --checkpoint /kaggle/working/runs/finetune/best.pt

In [ ]:
# Bundle everything to hand back: checkpoints, logs, metrics, onnx, metadata
!mkdir -p /kaggle/working/handoff
!cp -r ml/results api/model_meta.json api/labels.json ml/classes.json \
    artifacts/model.onnx artifacts/checksums.json \
    /kaggle/working/runs/finetune/best.pt /kaggle/working/runs/*/history.csv \
    /kaggle/working/handoff/ 2>/dev/null; ls -lh /kaggle/working/handoff